# Guide v3 — Nhận diện Vật cản 3D & Dẫn đường Đa mô hình trên Kaggle GPU

Notebook này chạy trực tiếp lõi `guide_core` thế hệ v3 trên GPU Kaggle (`Tesla T4` hoặc **`GPU T4 x2`** song song 2 GPU), bao gồm:
- **Nhận diện đa tầng**: `YOLO26n ROD` (`rod_26`, Recall 89.8%) hoặc `YOLOv8n ROD` (`rod_v8`) + `SegFormer B0 ADE20K` (giữ nguyên tỷ lệ khung hình, gộp xác suất biên độ phân giải 1/4) + `Depth Anything V2 Small` (khớp mặt phẳng đất RANSAC phát hiện vật cản nổi & hố sụt/bậc thềm).
- **Quy hoạch hành lang an toàn**: Multi-Horizon Dynamic Programming có bù chuyển động camera (Ego-motion EMA), cảnh báo hướng đồng hồ (`10h–2h`), phát hiện vật cản ngang đầu (`zone='head'`) và thời gian va chạm Looming `TTC`.
- **Điều phối Âm thanh Thông minh 4 Tầng (`Smart Arbitration`)**: P4 `STOP` cắt khẩn cấp `0 ms` kèm khóa an toàn `0.8s`; P3 cắt lệnh rẽ bị mâu thuẫn/thu hồi kèm earcon báo hủy; P2/P1 đọc tròn câu và gộp lệnh mới nhất vào hộp thư đơn (0 chồng tiếng), xuất âm thanh nổi Stereo.

**Chạy ngay không cần upload lại 117 MB trọng số:** Chỉ cần Import file `.ipynb` này lên Kaggle, bật **Accelerator = GPU T4 x2**, **Internet = ON**, gắn Dataset **`guide-v2-models`** sẵn có (hoặc `guide-v3-models`) cùng Dataset chứa video `.mp4`/`.mov`, rồi bấm **Run All**.

## 0. Khám phá dữ liệu Input trên Kaggle

In [ ]:
import os
from pathlib import Path

os.environ['YOLO_AUTOINSTALL'] = 'false'
INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists(), 'Notebook này được thiết kế để chạy trong môi trường Kaggle (/kaggle/input).'

print('=== Các thư mục trong /kaggle/input ===')
for p in sorted(INPUT_ROOT.iterdir()):
    print(' -', p)

FOUND_MANIFESTS = sorted(INPUT_ROOT.rglob('bundle_manifest.json'))
FOUND_ZIPS = sorted(INPUT_ROOT.rglob('guide_kaggle_*.zip'))
FOUND_VIDEOS = sorted({
    p for ext in ('*.mp4', '*.MP4', '*.mov', '*.MOV', '*.avi', '*.mkv')
    for p in INPUT_ROOT.rglob(ext)
})

print('\n=== Gói mô hình / Manifest tìm thấy ===')
for p in FOUND_MANIFESTS + FOUND_ZIPS:
    print(' -', p)

print('\n=== Video tìm thấy ===')
for p in FOUND_VIDEOS:
    print(' -', p)

## 1. Giải nén gói mô hình & Kiểm tra toàn vẹn SHA-256

Ô dưới tự động tìm gói `bundle_manifest.json` hoặc file `guide_kaggle_*models*.zip` trong `/kaggle/input` nếu bạn để trống `BUNDLE_SOURCE`. Nếu bạn gắn đồng thời Dataset mô hình v2 cũ và file `guide_kaggle_v3_code_only.zip`, ô này sẽ tự động cập nhật mã nguồn v3 lên trên trọng số sẵn có.

In [ ]:
import hashlib
import json
import shutil
import zipfile

BUNDLE_SOURCE = ''  # Để trống để tự nhận diện, hoặc điền đường dẫn cụ thể từ ô trên
CODE_OVERLAY_SOURCE = ''  # Tùy chọn: đường dẫn guide_kaggle_v3_code_only.zip nếu dùng chung với Dataset model v2 cũ
WORK = Path('/kaggle/working/guide_v3')

def resolve_bundle_source(explicit_path):
    if explicit_path:
        p = Path(explicit_path)
        assert p.exists(), f'Không tìm thấy BUNDLE_SOURCE: {p}'
        return p
    preferred = Path('/kaggle/input/datasets/nguyenduchng/guide-v2-models')
    if (preferred / 'bundle_manifest.json').is_file():
        return preferred
    model_manifests = [m.parent for m in FOUND_MANIFESTS if (m.parent / 'models').exists()]
    if len(model_manifests) == 1:
        return model_manifests[0]
    model_zips = [z for z in FOUND_ZIPS if 'code' not in z.name.lower()]
    if len(model_zips) == 1:
        return model_zips[0]
    if len(FOUND_MANIFESTS) == 1:
        return FOUND_MANIFESTS[0].parent
    raise AssertionError(f'Hãy điền BUNDLE_SOURCE rõ ràng. Ứng viên tìm thấy: {FOUND_MANIFESTS + FOUND_ZIPS}')

def unpack_and_verify(source, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        with zipfile.ZipFile(source) as z:
            for name in z.namelist():
                target = (dest_dir / name).resolve()
                assert target.is_relative_to(dest_dir.resolve()) and '\\' not in name, f'Unsafe ZIP path: {name}'
            z.extractall(dest_dir)
        manifest_data = json.loads(zipfile.ZipFile(source).read('bundle_manifest.json'))
    else:
        manifest_data = json.loads((source / 'bundle_manifest.json').read_text(encoding='utf-8'))
        for item in manifest_data['files']:
            name = item['path']
            target = (dest_dir / name).resolve()
            assert target.is_relative_to(dest_dir.resolve()) and '\\' not in name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source / name, target)
        shutil.copy2(source / 'bundle_manifest.json', dest_dir / 'bundle_manifest.json')
    for item in manifest_data['files']:
        p = (dest_dir / item['path']).resolve()
        assert p.is_relative_to(dest_dir.resolve())
        digest = hashlib.sha256()
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                digest.update(chunk)
        assert digest.hexdigest() == item['sha256'], f'Sai lệch SHA-256 tại: {p}'
    return manifest_data

resolved_bundle = resolve_bundle_source(BUNDLE_SOURCE)
print('Đang nạp gói chính từ:', resolved_bundle)
manifest = unpack_and_verify(resolved_bundle, WORK)

code_candidates = [Path(CODE_OVERLAY_SOURCE)] if CODE_OVERLAY_SOURCE else [
    z for z in FOUND_ZIPS if 'v3' in z.name.lower() and 'code' in z.name.lower()
]
for overlay in code_candidates:
    if overlay.exists() and overlay.resolve() != resolved_bundle.resolve():
        print('Đang cập nhật mã nguồn v3 từ:', overlay)
        unpack_and_verify(overlay, WORK)

os.chdir(WORK)
print(f'Kiểm tra toàn vẹn SHA-256 hoàn tất ({len(manifest["files"])} tệp). Thư mục làm việc: {WORK}')

# --- Tự động nạp bản nâng cấp mã nguồn Guide v3 (kiểm tra SHA-256 từng tệp) ---
import base64 as _b64, zlib as _zlib
_V3_ARCHIVE_SHA256 = 'fe586bc094fed9b0fac3ded94adc774c2b78e348f63243dd0dd6fc12bc2f43b6'
_V3_COMPRESSED_B64 = (
    '...'
)
_compressed_bytes = _b64.b64decode(_V3_COMPRESSED_B64)
assert hashlib.sha256(_compressed_bytes).hexdigest() == _V3_ARCHIVE_SHA256, 'Lỗi toàn vẹn gói overlay Guide v3'
_v3_files = json.loads(_zlib.decompress(_compressed_bytes).decode('utf-8'))
for _rel_path, _entry in _v3_files.items():
    _raw = _b64.b64decode(_entry['b64'])
    assert hashlib.sha256(_raw).hexdigest() == _entry['sha256'], f'Lỗi SHA-256 overlay: {_rel_path}'
    _dest = (WORK / _rel_path).resolve()
    assert _dest.is_relative_to(WORK.resolve())
    _dest.parent.mkdir(parents=True, exist_ok=True)
    _dest.write_bytes(_raw)
print(f'Đã nạp & xác thực SHA-256 thành công {len(_v3_files)} tệp mã nguồn Guide v3 mới nhất!')


## 2. Kiểm tra & Cài đặt thư viện (Bảo toàn PyTorch CUDA của Kaggle)

Ô này kiểm tra các gói cần thiết trước khi gọi `pip`, đồng thời khóa cứng phiên bản `torch` / `torchvision` / `torchaudio` sẵn có trên Kaggle để giữ nguyên driver CUDA.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

REQUIRED_EXACT = {
    'ultralytics': '8.4.178',
    'transformers': '5.20.0',
    'opencv-python': '4.14.0.94',
    'imageio-ffmpeg': '0.6.0',
}

needs_install = False
for pkg, expected_ver in REQUIRED_EXACT.items():
    try:
        if metadata.version(pkg) != expected_ver:
            needs_install = True
    except metadata.PackageNotFoundError:
        needs_install = True
try:
    metadata.version('lap')
except metadata.PackageNotFoundError:
    needs_install = True

if needs_install:
    constraints = []
    for package in ('torch', 'torchvision', 'torchaudio'):
        try:
            constraints.append(f'{package}=={metadata.version(package)}')
        except metadata.PackageNotFoundError:
            pass
    Path('kaggle_torch_constraints.txt').write_text('\n'.join(constraints), encoding='utf-8')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--upgrade-strategy', 'only-if-needed',
        '-c', 'kaggle_torch_constraints.txt',
        '-r', 'requirements-guide-core.txt',
    ], check=True)
else:
    print('Tất cả thư viện đã đúng phiên bản chuẩn, bỏ qua cài đặt lại.')

import torch
assert torch.cuda.is_available(), 'Chưa bật GPU trên Kaggle! Hãy chọn Settings -> Accelerator -> GPU T4 x2.'
GPU_COUNT = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__} | Số lượng GPU khả dụng: {GPU_COUNT}')
for idx in range(GPU_COUNT):
    print(f' - cuda:{idx}: {torch.cuda.get_device_name(idx)}')

## 3. Cấu hình Video, Bối cảnh, Song song Đa GPU & Âm thanh 4 Tầng

- **`PRIMARY`**: `'rod_26'` (khuyên dùng — YOLO26n ROD đạt Recall `89.8%`, mAP50 `93.1%`) hoặc `'rod_v8'` (YOLOv8n ROD).
- **`ENVIRONMENT`**: `'auto'` (mặc định — tự động nhận biết trong nhà `indoor` / vỉa hè `sidewalk` / ngõ hẻm `alley` từ xác suất ngữ nghĩa), hoặc ép cứng `'indoor'` / `'sidewalk'` / `'alley'`.
- **`MODE`**: `'offline'` (phân tích chuẩn từng khung hình ở `sample_fps=10`) hoặc `'realtime_replay'` (mô phỏng luồng camera thời gian thực, đo độ trễ `frame_to_decision_ms` và cơ chế bỏ khung trễ).
- **`MULTI_GPU`**: Khi bật `GPU T4 x2` trên Kaggle, tự động tách Detector sang `cuda:0` và SegFormer + Depth Anything V2 sang `cuda:1` để chạy song song không nghẽn cổng thiết bị.

In [ ]:
from guide_core.config import Config

VIDEO_PATH = ''          # Điền đường dẫn .mp4/.mov cụ thể nếu gắn nhiều video trong /kaggle/input
PRIMARY = 'rod_26'       # 'rod_26' (khuyên dùng) hoặc 'rod_v8'
SECOND_DETECTOR = False  # Bật cả 2 YOLO cùng lúc (chỉ dùng khi thử nghiệm đối chiếu)
ENVIRONMENT = 'auto'     # 'auto' (mặc định tự nhận biết) | 'indoor' | 'sidewalk' | 'alley'
MODE = 'offline'         # 'offline' | 'realtime_replay'
SAMPLE_FPS = 10          # Tần số lấy mẫu khung hình cho chế độ offline
MAX_SECONDS = None       # Đặt số giây (vd: 10) để chạy thử nhanh, hoặc None để chạy hết video
MULTI_GPU = True         # Tận dụng song song 2 GPU khi chọn Kaggle GPU T4 x2
SMART_ARBITRATION = True # Điều phối âm thanh thông minh 4 tầng + earcon + stereo

bundled_video = manifest.get('video')
if VIDEO_PATH:
    VIDEO = Path(VIDEO_PATH)
elif bundled_video and (WORK / bundled_video).is_file():
    VIDEO = WORK / bundled_video
elif len(FOUND_VIDEOS) == 1:
    VIDEO = FOUND_VIDEOS[0]
    print('Tự động chọn video duy nhất trong Input:', VIDEO)
else:
    raise AssertionError(f'Hãy điền VIDEO_PATH bằng 1 trong các video sau: {[str(v) for v in FOUND_VIDEOS]}')

assert VIDEO.is_file(), f'Không tìm thấy file video: {VIDEO}'
assert PRIMARY in {'rod_v8', 'rod_26'}

base_cfg_path = Path('configs/guide_optimized.json') if Path('configs/guide_optimized.json').is_file() else Path('configs/guide_kaggle.json')
config = json.loads(base_cfg_path.read_text(encoding='utf-8'))
config.update(
    environment=ENVIRONMENT,
    mode=MODE,
    mirror_input=False,
    sample_fps=SAMPLE_FPS,
    confirm_horizon_ratio=0.70,
    temporal_ema_alpha=0.65,
    ttc_stop_s=1.6,
    head_hazard_enabled=True,
    depth_obstacle_enabled=True,
    preempt_emergency_audio=SMART_ARBITRATION,
    smart_arbitration=SMART_ARBITRATION,
    post_stop_hold_s=0.8,
    post_stop_turn_hold_s=0.3,
    tail_grace_s=0.15,
    revocation_debounce_frames=2,
    earcon_enabled=SMART_ARBITRATION,
    stereo_audio=SMART_ARBITRATION,
    audio_enabled=True,
)

context_device = 'cuda:1' if (MULTI_GPU and GPU_COUNT >= 2) else 'cuda:0'
for m in config['models']:
    if m['role'] == 'detector':
        m['device'] = 'cuda:0'
        m['enabled'] = (m['id'] == PRIMARY) or SECOND_DETECTOR
    else:
        m['device'] = context_device

Path('selected_config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
validated = Config.read('selected_config.json')
print('Video đầu vào :', VIDEO)
print('Cấu hình gốc  :', base_cfg_path)
print('Phân bổ thiết bị:', {m.id: m.device for m in validated.models if m.enabled})

## 4. Kiểm tra tương thích ba mô hình trên một khung hình thực

Ô này nạp toàn bộ mô hình đang bật lên GPU và chạy thử trên khung hình đầu tiên của video để đảm bảo `failures = {}` trước khi chạy toàn bộ video.

In [ ]:
subprocess.run([
    sys.executable, 'tools/check_guide_environment.py',
    '--config', 'selected_config.json',
    '--video', str(VIDEO),
    '--report', '/kaggle/working/compatibility.json',
], check=True)

## 5. Chạy nhận diện & Xuất video dẫn đường `demo.mp4`

In [ ]:
from datetime import datetime

RUN = Path('/kaggle/working') / ('guide_v3_' + VIDEO.stem + '_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
command = [
    sys.executable, '-m', 'guide_core.demo',
    '--config', 'selected_config.json',
    '--video', str(VIDEO),
    '--output', str(RUN),
]
if MAX_SECONDS is not None:
    command += ['--seconds', str(MAX_SECONDS)]

subprocess.run(command, check=True)
report = json.loads((RUN / 'report.json').read_text(encoding='utf-8'))
print('Đã xuất video demo :', RUN / 'demo.mp4')
print(f'Tốc độ suy luận    : {report["unique_inference_frames_per_second"]:.2f} FPS | Thời gian xử lý: {report["processing_wall_s"]:.2f}s')
print('Phân bố hành động  :', report['action_counts'])
print('VRAM đỉnh theo GPU :', report.get('cuda_peak_allocated_by_device_mb') or report.get('cuda_peak_allocated_mb'))

In [ ]:
from IPython.display import Video, display

display(Video(str(RUN / 'demo.mp4'), embed=True, width=760))

## 6. Trực quan hóa Độ trễ GPU, Dòng thời gian Quyết định & Lịch Âm thanh 4 Tầng

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

decisions = [json.loads(line) for line in (RUN / 'decisions.jsonl').read_text(encoding='utf-8').splitlines()]
segments = json.loads((RUN / 'speech_segments.json').read_text(encoding='utf-8'))

ACTION_COLORS = {
    'STOP': '#d62728',
    'SHIFT_LEFT': '#1f77b4',
    'SHIFT_RIGHT': '#ff7f0e',
    'KEEP_HEADING': '#2ca02c',
    'SLOW': '#bcbd22',
    'UNCLEAR': '#7f7f7f',
}

fig, axes = plt.subplots(3, 1, figsize=(14, 9.5), constrained_layout=True)

# 1. Độ trễ suy luận theo từng mô hình (p50 vs p95 ms)
ax0 = axes[0]
model_names = list(report['model_ms'].keys())
p50_vals = [report['model_ms'][m]['p50'] or 0.0 for m in model_names]
p95_vals = [report['model_ms'][m]['p95'] or 0.0 for m in model_names]
y_pos = np.arange(len(model_names))
bar_h = 0.36
ax0.barh(y_pos - bar_h / 2, p50_vals, height=bar_h, color='#1f77b4', label='Trung vị p50 (ms)')
ax0.barh(y_pos + bar_h / 2, p95_vals, height=bar_h, color='#ff7f0e', label='Đuôi p95 (ms)')
ax0.set_yticks(y_pos)
ax0.set_yticklabels(model_names)
ax0.set_xlabel('Thời gian suy luận trên GPU (ms)')
ax0.set_title('Độ trễ suy luận từng mô hình trên Kaggle GPU')
ax0.legend(loc='lower right')
ax0.grid(axis='x', linestyle='--', alpha=0.35)

# 2. Chuỗi thời gian độ trễ xử lý khung hình liên tục
ax1 = axes[1]
times_s = [d['video_time_s'] if report['mode'] == 'offline' else d['emitted_at_s'] for d in decisions]
if report['mode'] == 'realtime_replay':
    latencies_ms = [d.get('frame_to_decision_ms') or 0.0 for d in decisions]
    series_label = 'Độ trễ Frame-to-Decision (ms)'
else:
    latencies_ms = [sum(m.get('ms') or 0.0 for m in d.get('models', [])) for d in decisions]
    series_label = 'Tổng thời gian mô hình mỗi khung (ms)'
ax1.plot(times_s, latencies_ms, color='#2ca02c', linewidth=1.6, label=series_label)
ax1.set_xlabel('Thời gian (giây)')
ax1.set_ylabel('Độ trễ (ms)')
ax1.set_title('Biến thiên độ trễ xử lý theo thời gian video')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.35)

# 3. Dải lịch phát câu thoại 4 tầng (kiểm chứng trực quan không chồng tiếng)
ax2 = axes[2]
seen_actions = set()
for seg in segments:
    act = seg['action']
    lbl = act if act not in seen_actions else None
    seen_actions.add(act)
    dur = max(0.02, seg['end'] - seg['start'])
    ax2.broken_barh([(seg['start'], dur)], (0.2, 0.6), facecolors=ACTION_COLORS.get(act, '#333333'), edgecolor='black', label=lbl)
ax2.set_ylim(0, 1)
ax2.set_yticks([])
ax2.set_xlim(0, max(report['export_duration_s'], 1.0))
ax2.set_xlabel('Thời gian xuất âm thanh (giây)')
ax2.set_title('Dải thời gian phát lệnh thoại 4 tầng (Speech Segments — 0 chồng lắp)')
if seen_actions:
    ax2.legend(loc='upper right', ncol=min(6, len(seen_actions)))
ax2.grid(axis='x', linestyle='--', alpha=0.35)
plt.show()

## 7. Kiểm định bất biến Âm thanh An toàn & Đóng gói ZIP tải về

Xác nhận hai bất biến cốt lõi của bộ điều phối âm thanh 4 tầng:
1. **Không bao giờ chồng tiếng** (`overlaps == 0`).
2. Mọi sự kiện cắt câu (`cancelled`) chỉ được phép xuất phát từ **P4 Khẩn cấp (`STOP`)** hoặc **P3 Mâu thuẫn / Thu hồi hướng rẽ**.

In [ ]:
events = [json.loads(line) for line in (RUN / 'audio_events.jsonl').read_text(encoding='utf-8').splitlines()]
allowed_preemptions = {
    'preempted_by_emergency_stop',
    'preempted_by_priority_direction',
    'preempted_contradicted_direction',
    'preempted_revoked_direction',
}

overlaps = sum(a['end'] > b['start'] + 1e-6 for a, b in zip(segments, segments[1:]))
cancelled = [e for e in events if e['event'] == 'cancelled' and e.get('reason') != 'stopped_or_muted']
unexpected = [e for e in cancelled if e.get('reason') not in allowed_preemptions]

assert overlaps == 0, f'Phát hiện chồng tiếng: {overlaps}'
assert not unexpected, f'Phát hiện lý do ngắt câu không hợp lệ: {unexpected}'

print(f'Tổng số đoạn phát thoại : {len(segments)} | Chồng tiếng: {overlaps}')
print(f'Số lần cắt khẩn cấp P4/P3: {len(cancelled)} | Chi tiết: {report["audio"]["interruptions_by_reason"]}')

archive = shutil.make_archive(str(RUN), 'zip', RUN)
print('Gói kết quả sẵn sàng tải tại tab Output:', archive)

## 8. So sánh Ablation các mô hình trên cùng khung hình (Tùy chọn)

Đặt `RUN_COMPARISON = True` nếu bạn muốn đo tốc độ và đối chiếu quyết định giữa `rod_v8`, `rod_26`, `+SegFormer`, `+Depth` và `2 Detectors` trên cùng tập khung hình.

In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    compare_dir = Path('/kaggle/working') / ('guide_compare_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    subprocess.run([
        sys.executable, 'tools/compare_guide_models.py',
        '--config', 'selected_config.json',
        '--video', str(VIDEO),
        '--output', str(compare_dir),
        '--seconds', '8',
        '--sizes', '480',
    ], check=True)
    print((compare_dir / 'comparison.json').read_text(encoding='utf-8'))

### Q&A
- **Làm sao chạy nhận diện nhanh nhất trên Kaggle mà không lỗi xung đột thư viện?** Khóa nguyên bản `torch` / `torchvision` / `torchaudio` của Kaggle bằng `kaggle_torch_constraints.txt`, chỉ cài bổ sung `ultralytics==8.4.178` và `transformers==5.20.0`, đồng thời bật `GPU T4 x2` để tách riêng luồng Detector (`cuda:0`) và SegFormer + Depth (`cuda:1`).
- **Khi nào câu lệnh thoại được đọc hết câu, khi nào bị cắt ngang?** Các câu dẫn đường thông thường (P2/P1) luôn được đọc hết câu và gộp lệnh mới nhất vào hộp thư đơn; chỉ lệnh sinh tồn `STOP` (P4, cắt `0 ms`) hoặc lệnh rẽ đang phát bị mâu thuẫn/thu hồi trước `0.15s` cuối câu (P3) mới được phép cắt ngang.

### Data Analysis Key Findings
- Báo cáo `report.json` ghi nhận đầy đủ phân phối độ trễ `p50` / `p95` (ms) của từng mô hình (`rod_26` / `rod_v8`, `ade20k_b0`, `depth_v2_small`), mức tiêu thụ VRAM theo từng GPU (`cuda:0`, `cuda:1`) và tỷ lệ từng hành động dẫn đường.
- Bộ kiểm định âm thanh xác nhận `overlaps == 0` trên toàn bộ `speech_segments.json` và thống kê minh bạch các lần cắt câu hợp lệ theo từng lý do an toàn trong `interruptions_by_reason`.

### Insights or Next Steps
- Thử chuyển `MODE = 'realtime_replay'` ở Mục 3 để đo trực tiếp độ trễ đầu-cuối `frame_to_decision_ms` và kiểm chứng cơ chế loại bỏ khung hình quá hạn trên GPU T4.
- Tải gói `.zip` trong mục Output về máy, dùng `tools/annotate_guide.html` gắn nhãn các đoạn nguy hiểm/thông thoáng rồi chấm điểm định lượng bằng `python -m guide_core.evaluate`.